# Sandbox extendido del pipeline sin malla -- eje + plano, 100% auto-contenido

Version extendida de `sandbox_pipeline_sin_malla.ipynb`: ahora corre **eje
(axis_v06) y plano (plane_v04_1) en paralelo**, cada uno sobre sus propios 30
objetos reales (10 BUENO / 10 MEDIO / 10 MALO), seleccionados con
`Mapping/select_sandbox_objects.py` a partir de `axis_v06_nomesh`
(por `angular_error_deg`) y `plane_v04_1_nomesh` (por F1 por-objeto
aproximado -- ver el docstring de ese script para por que `f1_ref` en si no
es una metrica por objeto).

```
1. Listas de objetos (reales, ya seleccionadas)
2. Copiar .obj/.txt y renders existentes -> carpeta propia en Experiments/
3. Prompts (axis_v06 y plane_v04_1, editables en celdas)
4. Geometria: camara, triangulacion (eje Y plano), filtro fondo/objeto
5. Molmo2: carga del modelo + inferencia (inline)          [necesita GPU]
6. Correr inferencia (eje y plano, por separado)
7. Estimacion sin malla (eje: triangulation; plano: triangulation_multiplane)
8. Evaluacion contra GT (eje y plano)
9. Resultados por objeto, con su categoria BUENO/MEDIO/MALO
10. Visualizacion 2D/3D (opcional)
```

**Ningun paso llama a un script .py del repo** -- todo el codigo esta
copiado inline (prompts, geometria, parseo de Molmo2, metricas) para poder
editarlo directo en las celdas. Es una copia funcional de
`MolmoPointing/molmo_multiview_runner.py`, `pipeline_common/camera.py`,
`pipeline_common/triangulation.py`, `Mapping/estimate_symmetry_no_mesh.py` y
`Mapping/evaluate.py` -- si una variante gana aca, hay que trasplantarla a
mano a esos archivos para que entre al sweep completo de 850 objetos.

**Incluye `SDE_ref`/`F1_ref`** (formulas exactas de `Mapping/evaluate.py`,
requieren `gpytoolbox` -- disponible en el servidor). `SDE_ref` aplica a eje
Y plano; `F1_ref` solo a plano (no existe una convencion de F1 para eje, ni
en el repo de referencia ni en la literatura). Se reportan tanto para el
conjunto completo como para cada subconjunto BUENO/MEDIO/MALO por separado.

Los `.obj`/`.txt` estan en `data/objects/curated_{axis,plane}_sym_obj/` y
los renders en `data/renders/{axis,plane}_sym/<object_id>/224/flat/` --
misma ruta `data/` que usan todos los scripts `.py` del repo (esto corre en
el servidor).

## 0. Setup

In [ ]:
import json
import re
import shutil
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import trimesh
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

REPO_ROOT = Path(r"C:\Users\HP\Desktop\Seminario de Tesis I\Symmetry-Detection-Using-Multimodal-Vision-Language-Models")
DATA_ROOT = Path(r"C:\Users\HP\Desktop\Seminario de Tesis I\data")  # solo lectura -- fuente de .obj/.txt/renders
# En el servidor, ajustar a algo como Path("../data") si corres desde la raiz del repo con rutas relativas.

SANDBOX_ROOT    = REPO_ROOT / "Experiments" / "sandbox_pipeline_sin_malla_extended_data"
SANDBOX_OBJECTS = SANDBOX_ROOT / "objects"
SANDBOX_RENDERS = SANDBOX_ROOT / "renders"

OBJECTS_SUBDIR_MAP = {"axis_sym": "curated_axis_sym_obj", "plane_sym": "curated_plane_sym_obj"}
SIZE, LIGHTING = 224, "flat"
DEFAULT_FOV    = 60.0
VIEW_GROUPS    = [6, 14, 26]

MODEL_ID = "allenai/Molmo2-8B"

EDGE_ON_THRESH_DEFAULT   = 0.5    # plano: |cos(angulo)| por debajo = vista "de canto"
DUP_ANGLE_THRESH_DEFAULT = 15.0   # plano: grados; planos candidatos mas cerca que esto = duplicados

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110})

## 1. Listas de objetos (reales, 10 BUENO / 10 MEDIO / 10 MALO cada uno)

Generadas con `python Mapping/select_sandbox_objects.py ...` sobre
`axis_v06_nomesh` (angular_error) y `plane_v04_1_nomesh` (F1 por-objeto
aproximado). Formato de entrada `{CATEGORIA}_{object_id}_2d`.

In [ ]:
def parse_entry(entry: str) -> tuple[str, str]:
    """'BUENO_89cb9b2ad175b833cadf6344ec272e8_2d' -> ('BUENO', '89cb9b2ad175b833cadf6344ec272e8')."""
    parts = entry.split("_")
    categoria = parts[0]
    object_id = next(p for p in parts[1:] if len(p) >= 30 and all(c in "0123456789abcdef" for c in p))
    return categoria, object_id


RAW_OBJECT_LIST_AXIS = [
    "BUENO_941271c5d9b192eaccd8f9b9403fd602_2d",
    "BUENO_e0725fd7859fa238ff67c12005f72d2_2d",
    "BUENO_89cb9b2ad175b833cadf6344ec272e8_2d",
    "BUENO_dc2c07ff617d1da0197a35146ee825cd_2d",
    "BUENO_b35973652d9a526c64848cfde3847b25_2d",
    "BUENO_28d39c9673775f601a3b47d6d0918049_2d",
    "BUENO_3a29634236aa0596f9e8cd846ef13776_2d",
    "BUENO_9fe7e6a7bf8ca964efad53eb3f0b36fa_2d",
    "BUENO_5e914dd726313181b1b4b514128408b0_2d",
    "BUENO_b2acbb6717c7a842fcb8d8c6d4df8143_2d",
    "MEDIO_8d457deaf22394da65c5c31ac688ec4_2d",
    "MEDIO_d3a3d52234e722825208aab875b932bc_2d",
    "MEDIO_94fdbb748526dae4ea2d70ab68cd1d2_2d",
    "MEDIO_297d929269bb62da43fdcbcacbbed64c_2d",
    "MEDIO_6b8b2cb01c376064c8724d5673a063a6_2d",
    "MEDIO_f0611ec9ff89209bf10c4513652c1c5e_2d",
    "MEDIO_33b77c66e1f849b790c4e2a44fddf755_2d",
    "MEDIO_1b4d7803a3298f8477bdcb8816a3fac9_2d",
    "MEDIO_d851cbc873de1c4d3b6eb309177a6753_2d",
    "MEDIO_618d55a791b8280cf256a8c3e3396495_2d",
    "MALO_64f61c9c81e3eb7b8aaae3d020f5ddf8_2d",
    "MALO_66cec0a2ab63d9101b6c273f8ff0e8b6_2d",
    "MALO_8adc0ce79962ac2021072d05c97a5e0a_2d",
    "MALO_6e1fe96adbb5ffba8bae2d07dadd1b5d_2d",
    "MALO_40aee078f0390b7134076b5960251711_2d",
    "MALO_56efabe64ce30e08d7bc2c778b186f58_2d",
    "MALO_5dd2f4d3b253058dd554ab0a45f30de7_2d",
    "MALO_6267ef99cbfaea7741cf86c757faf4f9_2d",
    "MALO_955143d7f0b5c70fef76898f881b76a_2d",
    "MALO_e656d6586d481f41eb69804478f9c547_2d",
]

RAW_OBJECT_LIST_PLANE = [
    "BUENO_109a19840a8fbc774c3aee8e9d3a6ffa_2d",
    "BUENO_1541e36e8dc2d84caed2201239784a35_2d",
    "BUENO_15e2d1fe9a4b28e68fb8c03013603b0c_2d",
    "BUENO_1655608678a4f9ffbc7bb5239e53ea6f_2d",
    "BUENO_1ad4c572e0fd6a576e1e9a13188ab4bb_2d",
    "BUENO_1b6a5fc808388138cb2a965e75be701c_2d",
    "BUENO_2620701a50216dbed0b36851d61b6fca_2d",
    "BUENO_2912f56d135e2f97b20cb946bceb58f_2d",
    "BUENO_2b25e49c58ae0e292056b4bd5d870b47_2d",
    "BUENO_3279edef6a631940ea41b93204b74265_2d",
    "MEDIO_82d8391c62e197161282d4d9178caa92_2d",
    "MEDIO_82e0e96bd89956d9f330eedffaf3339e_2d",
    "MEDIO_835e01af50296235aefda81565fede7_2d",
    "MEDIO_8399987b9bca0dd0e63a9e8397b31118_2d",
    "MEDIO_83c90f7b104816ecc748af2814b558c4_2d",
    "MEDIO_8497e02fa1662113776d8bc79b9caa2c_2d",
    "MEDIO_84aa911799cd87b4ad5067eac75a07f7_2d",
    "MEDIO_84efcf2796fad0d2917fe9209d09e56e_2d",
    "MEDIO_854ffe2c60091876e07a1c4b84dfd325_2d",
    "MEDIO_8596664f3d7925cdfdeb515ad63cf4b0_2d",
    "MALO_a4b457c41180e0b9fe52ffd0e748a1ab_2d",
    "MALO_a85b9a3ace923e424721d5612f98ae26_2d",
    "MALO_ac52cf0b598e930ab38d3c03866c1379_2d",
    "MALO_ad86354fb5faf1c98a4820926b2a786_2d",
    "MALO_d0c3bf270f9e04eb362845c6edb57fc_2d",
    "MALO_d2553e5fc4f1527cfeae521e94848af6_2d",
    "MALO_d2a511578a387365ede9471c962dd6ba_2d",
    "MALO_dacf794f991fc0173e9b7c3ab7636200_2d",
    "MALO_dd2a4c416625f29c4f57a7ededfb3bde_2d",
    "MALO_de519752147a225032387cdb9b2a84d5_2d",
]

OBJECT_LIST_AXIS  = [parse_entry(e) for e in RAW_OBJECT_LIST_AXIS]
OBJECT_LIST_PLANE = [parse_entry(e) for e in RAW_OBJECT_LIST_PLANE]
CATEGORY_BY_ID_AXIS  = {oid: cat for cat, oid in OBJECT_LIST_AXIS}
CATEGORY_BY_ID_PLANE = {oid: cat for cat, oid in OBJECT_LIST_PLANE}

print(f"axis_sym : {len(OBJECT_LIST_AXIS)} objetos")
print(f"plane_sym: {len(OBJECT_LIST_PLANE)} objetos")

## 2. Copiar datos al sandbox

Solo copia lo minimo: `.obj`/`.txt` (GT) y, de cada render ya generado, el
`manifest.json` + `metadata_all.json` (poses de camara) + los PNG -- **sin
volver a renderizar**. No copia ningun `molmo_multiview*`/`predicted_*`/
`eval_*` -- eso se genera de cero en el sandbox.

In [ ]:
def copy_object_data(symmetry_type: str, object_list: list[tuple[str, str]]) -> None:
    objects_subdir = OBJECTS_SUBDIR_MAP[symmetry_type]
    sandbox_objects_dir = SANDBOX_OBJECTS / objects_subdir
    sandbox_objects_dir.mkdir(parents=True, exist_ok=True)

    for cat, oid in object_list:
        src_objects_dir = DATA_ROOT / "objects" / objects_subdir
        for ext in (".obj", ".txt"):
            src, dst = src_objects_dir / f"{oid}{ext}", sandbox_objects_dir / f"{oid}{ext}"
            if not dst.exists():
                assert src.exists(), f"No encontrado: {src}"
                shutil.copy2(src, dst)

        src_render_dir = DATA_ROOT / "renders" / symmetry_type / oid / str(SIZE) / LIGHTING
        dst_render_dir = SANDBOX_RENDERS / symmetry_type / oid / str(SIZE) / LIGHTING
        dst_render_dir.mkdir(parents=True, exist_ok=True)
        assert src_render_dir.exists(), f"No encontrado: {src_render_dir}"

        for meta_file in ("manifest.json", "metadata_all.json"):
            src_meta, dst_meta = src_render_dir / meta_file, dst_render_dir / meta_file
            if src_meta.exists() and not dst_meta.exists():
                shutil.copy2(src_meta, dst_meta)

        n_copied = 0
        for png in src_render_dir.glob("*.png"):
            dst_png = dst_render_dir / png.name
            if not dst_png.exists():
                shutil.copy2(png, dst_png)
                n_copied += 1
        print(f"[{symmetry_type:9s}][{cat:6s}] {oid}: {n_copied} PNG copiados")


copy_object_data("axis_sym", OBJECT_LIST_AXIS)
copy_object_data("plane_sym", OBJECT_LIST_PLANE)
print(f"\nSandbox listo en: {SANDBOX_ROOT}")

## 3. Prompts (editables aca, sin tocar archivos .txt)

`axis_v06` (mejor eje) y `plane_v04_1` (mejor plano por F1) -- copia
funcional de `MolmoPointing/prompts/{axis,plane}/...`.

In [ ]:
PROMPT_ID_AXIS = "axis_v06"

PROMPT_SINGLE_AXIS = """You are given ONE image of a 3D object.

The object has ONE dominant rotational symmetry axis.

Your task is to identify the two poles of the rotation axis: the topmost and bottommost points where the axis exits the object's surface.

Return:
- obj_id 1: the TOP pole -- topmost visible point on the rotation axis (at the horizontal center of the topmost surface)
- obj_id 2: the BOTTOM pole -- bottommost visible point on the rotation axis (at the horizontal center of the bottommost surface)

IMPORTANT RULES:
- Both points MUST lie ON the rotation axis -- at the horizontal center of the object at that height, NOT on the lateral silhouette.
- obj_id 1 MUST be above obj_id 2 (smaller Y value).
- The two points MUST be as far apart vertically as possible.
- Both points MUST lie on the visible object surface.
- For flat-topped or flat-bottomed objects, place the pole at the geometric center of the top or bottom face.
- For ROUNDED or CURVED tops/bottoms (no single flat face -- e.g. a dome, a sphere cap, a rounded knob), place the pole at the center of curvature of that rounded region: the point on the visible surface that is equidistant from the left and right silhouette edges at that region's topmost/bottommost extent.
- Verify: draw an imaginary horizontal line through each point -- the point should be equidistant from the left and right edges at that height, regardless of whether the surface there is flat or curved.
- Do NOT place points on the lateral silhouette edges.

Output ONLY:

<points coords="1 1 X1 Y1 2 X2 Y2">"""


PROMPT_MULTI_AXIS = """You are given multiple views of the SAME 3D object.

The object has ONE dominant axis of rotational symmetry.

For each image, identify the TOP and BOTTOM poles of the global rotation axis -- the points where the axis exits the object's surface at the very top and very bottom.

For each image:
1. Locate where the global axis exits at the top and bottom of the object.
2. Return:
   - obj_id 1: the TOP pole (topmost point on the axis, at the horizontal center of the topmost surface),
   - obj_id 2: the BOTTOM pole (bottommost point on the axis, at the horizontal center of the bottommost surface).

IMPORTANT RULES:
- Both points MUST be ON the rotation axis -- at the horizontal center of the object at that height, NOT on the lateral silhouette.
- Use the SAME global axis consistently across all views.
- obj_id 1 MUST be above obj_id 2 in each image.
- For flat surfaces (e.g., flat-topped objects), place the pole at the geometric center of the top or bottom face.
- For ROUNDED or CURVED tops/bottoms (no single flat face -- e.g. a dome, a sphere cap, a rounded knob), place the pole at the center of curvature of that rounded region: the point equidistant from the left and right silhouette edges at that region's topmost/bottommost extent.
- Infer the global axis from ALL views jointly before answering.
- Verify consistency: the TOP pole should correspond to the same geometric point on the object across all views, whether the surface there is flat or curved.
- Do NOT place points on the lateral silhouette edges.

Output format (one entry per image, separated by semicolons):

<points coords="1 1 Xtop Ytop 2 Xbottom Ybottom; 2 1 Xtop Ytop 2 Xbottom Ybottom; 3 1 Xtop Ytop 2 Xbottom Ybottom">

Where each entry is: image_index obj_id X Y
- obj_id 1 = TOP pole of the rotation axis
- obj_id 2 = BOTTOM pole of the rotation axis

Return ONLY the <points ...> block."""

print(f"Prompt eje cargado: {PROMPT_ID_AXIS}")

In [ ]:
PROMPT_ID_PLANE = "plane_v04_1"

PROMPT_SINGLE_PLANE = """You are given ONE image of a 3D object.

The object has ONE dominant plane of reflective symmetry.

Your task is to identify the horizontal midpoints of the object's left-right extent at two different heights: one near the TOP and one near the BOTTOM of the object.

These midpoints lie on the symmetry plane's projected trace.

Return:
- obj_id 1: the horizontal midpoint of the left-right extent near the TOP of the object
- obj_id 2: the horizontal midpoint of the left-right extent near the BOTTOM of the object

The horizontal midpoint at height Y = (X_left_edge + X_right_edge) / 2 at that height.

IMPORTANT RULES:
- Both points MUST lie on the symmetry plane trace (horizontal center of the object at that height).
- obj_id 1 MUST be in the upper half of the object's visible extent.
- obj_id 2 MUST be in the lower half of the object's visible extent.
- Both points MUST lie on or within the visible object.
- Do NOT place points on the silhouette edges -- place them at the HORIZONTAL CENTER between the edges.
- Do NOT collapse both points to the same height.
- Use the object's bilateral geometry to verify that the chosen midpoints are consistent with the symmetry plane direction.

Output ONLY:

<points coords="1 1 X1 Y1 2 X2 Y2">"""


PROMPT_MULTI_PLANE = """You are given multiple views of the SAME 3D object.

The object has ONE dominant plane of reflective symmetry.

Step 1: Identify the global symmetry plane from ALL views together -- it divides the object into two mirror halves.
Step 2: For each image, find the horizontal midpoints of the object's left-right extent at two heights (top and bottom) that lie on the plane's projected trace.

For each image:
1. Find the horizontal midpoint at the top of the object and at the bottom.
2. Return:
   - obj_id 1: horizontal midpoint near the TOP (upper half),
   - obj_id 2: horizontal midpoint near the BOTTOM (lower half).

The horizontal midpoint at height Y = (X_left_edge + X_right_edge) / 2 at that height.

IMPORTANT RULES:
- Use the SAME global symmetry plane consistently across all views.
- Both points in each image must be on the plane's projected trace (horizontal center of the object at that height).
- obj_id 1 MUST be above obj_id 2 in every image.
- Both points MUST lie within the visible object bounds.
- Do NOT place points on the silhouette edges -- only on the horizontal center between them.
- Infer the global plane from ALL views jointly before answering.
- Verify cross-view consistency: the midpoint direction across images should reflect the same global plane orientation.

Output format (one entry per image, separated by semicolons):

<points coords="1 1 Xtop Ytop 2 Xbottom Ybottom; 2 1 Xtop Ytop 2 Xbottom Ybottom; 3 1 Xtop Ytop 2 Xbottom Ybottom">

Where each entry is: image_index obj_id X Y
- obj_id 1 = horizontal midpoint near the TOP of the object
- obj_id 2 = horizontal midpoint near the BOTTOM of the object

Return ONLY the <points ...> block."""

print(f"Prompt plano cargado: {PROMPT_ID_PLANE}")

## 4. Geometria: camara, triangulacion (eje Y plano), filtro fondo/objeto

Copia funcional de `pipeline_common/camera.py` + `pipeline_common/triangulation.py`
+ la parte de plano de `Mapping/estimate_symmetry_no_mesh.py`.

In [ ]:
# --- pipeline_common/camera.py ---

def molmo_to_ndc(x: float, y: float) -> tuple[float, float]:
    ndc_x = (x / 1000.0) * 2.0 - 1.0
    ndc_y = 1.0 - (y / 1000.0) * 2.0
    return ndc_x, ndc_y


def build_camera_rays(ndc_x: float, ndc_y: float, R: list, T: list,
                      fov_deg: float, image_size: int) -> tuple[np.ndarray, np.ndarray]:
    """PyTorch3D row-vector convention: p_cam = p_world @ R + T, camera center = -(R @ T)."""
    R_np, T_np = np.array(R, dtype=np.float64), np.array(T, dtype=np.float64)
    ray_origin = -(R_np @ T_np)
    half_tan = np.tan(np.deg2rad(fov_deg) / 2.0)
    dir_cam = np.array([ndc_x * half_tan, ndc_y * half_tan, 1.0], dtype=np.float64)
    dir_world = R_np @ dir_cam
    dir_world /= np.linalg.norm(dir_world)
    return ray_origin, dir_world


# --- pipeline_common/triangulation.py ---

def ray_dir_for_point(x: float, y: float, R: list, T: list, fov_deg: float, image_size: int):
    ndc_x, ndc_y = molmo_to_ndc(x, y)
    return build_camera_rays(ndc_x, ndc_y, R, T, fov_deg, image_size)


def view_forward_direction(R: list, T: list, fov_deg: float, image_size: int) -> np.ndarray:
    _, direction = build_camera_rays(0.0, 0.0, R, T, fov_deg, image_size)
    return direction


def interpretation_plane_normal(dir_a: np.ndarray, dir_b: np.ndarray):
    n = np.cross(dir_a, dir_b)
    norm = np.linalg.norm(n)
    return None if norm < 1e-9 else n / norm


def triangulate_line(camera_centers: list, plane_normals: list) -> tuple[np.ndarray, np.ndarray]:
    N = np.asarray(plane_normals, dtype=np.float64)
    C = np.asarray(camera_centers, dtype=np.float64)
    _, _, Vt = np.linalg.svd(N)
    direction = Vt[-1]
    direction /= np.linalg.norm(direction)
    b = np.einsum("ij,ij->i", N, C)
    point, *_ = np.linalg.lstsq(N, b, rcond=None)
    return point, direction


def widest_pair(pts: list):
    """Par mas separado en pixeles entre TODOS los puntos de una vista -- None si hay <2."""
    if len(pts) < 2:
        return None
    best_pair, best_dist_sq = None, -1.0
    for i in range(len(pts)):
        for j in range(i + 1, len(pts)):
            dx, dy = pts[i]["x"] - pts[j]["x"], pts[i]["y"] - pts[j]["y"]
            dist_sq = dx * dx + dy * dy
            if dist_sq > best_dist_sq:
                best_dist_sq, best_pair = dist_sq, (pts[i], pts[j])
    return best_pair


def get_point_by_obj_id(pts: list, obj_id: int):
    return next((p for p in pts if p["obj_id"] == obj_id), None)

In [ ]:
# --- filtro fondo/objeto (Mapping/estimate_symmetry_no_mesh.py::filter_points_on_object) ---

BACKGROUND_THRESH = 250  # RGB > esto en los 3 canales = fondo blanco (PyTorch3D HardFlatShader default)


def is_on_object(pixel: np.ndarray) -> bool:
    return not bool(np.all(pixel[:3] > BACKGROUND_THRESH))


def molmo_xy_to_pixel(x: float, y: float, img_w: int, img_h: int) -> tuple[int, int]:
    px = int(round((x / 1000.0) * img_w))
    py = int(round((y / 1000.0) * img_h))
    return min(max(px, 0), img_w - 1), min(max(py, 0), img_h - 1)


def filter_points_on_object(points_by_image: dict, images_sent: list, render_dir: Path,
                            min_points: int = 2, image_cache: dict | None = None) -> dict:
    if image_cache is None:
        image_cache = {}
    filtered = {}
    for img_idx_str, pts in points_by_image.items():
        if not pts:
            filtered[img_idx_str] = pts
            continue
        cam = images_sent[int(img_idx_str)]
        filename = cam["filename"]
        if filename not in image_cache:
            img_path = render_dir / filename
            image_cache[filename] = np.array(Image.open(img_path).convert("RGB")) if img_path.exists() else None
        img = image_cache[filename]
        if img is None:
            filtered[img_idx_str] = pts
            continue
        img_h, img_w = img.shape[0], img.shape[1]
        kept = []
        for p in pts:
            px, py = molmo_xy_to_pixel(p["x"], p["y"], img_w, img_h)
            if is_on_object(img[py, px]):
                kept.append(p)
        filtered[img_idx_str] = kept if len(kept) >= min_points else []
    return filtered


# --- eje: Mapping/estimate_symmetry_no_mesh.py::estimate_axis_no_mesh ---

def estimate_axis_no_mesh(points_by_image: dict, images_sent: list, fov_deg: float, image_size: int):
    centers, normals = [], []
    for img_idx_str, pts in points_by_image.items():
        pair = widest_pair(pts)
        if pair is None:
            continue
        p_a, p_b = pair
        cam = images_sent[int(img_idx_str)]
        C, d_a = ray_dir_for_point(p_a["x"], p_a["y"], cam["R"], cam["T"], fov_deg, image_size)
        _, d_b = ray_dir_for_point(p_b["x"], p_b["y"], cam["R"], cam["T"], fov_deg, image_size)
        n = interpretation_plane_normal(d_a, d_b)
        if n is None:
            continue
        centers.append(C)
        normals.append(n)
    if len(normals) < 2:
        raise ValueError(f"need >=2 valid views, got {len(normals)}")
    point, direction = triangulate_line(centers, normals)
    return {"direction": direction.tolist(), "origin": point.tolist(), "n_views_used": len(normals)}

In [ ]:
# --- plano: Mapping/estimate_symmetry_no_mesh.py::_line_from_view_pair / estimate_plane_no_mesh / detect_planes_no_mesh ---

def _line_from_view_pair(view_i: int, view_j: int, points_by_image: dict, images_sent: list,
                         fov_deg: float, image_size: int):
    centers, normals = [], []
    for idx in (view_i, view_j):
        pts = points_by_image.get(str(idx), [])
        p1, p2 = get_point_by_obj_id(pts, 1), get_point_by_obj_id(pts, 2)
        if p1 is None or p2 is None:
            return None
        cam = images_sent[idx]
        C, d1 = ray_dir_for_point(p1["x"], p1["y"], cam["R"], cam["T"], fov_deg, image_size)
        _, d2 = ray_dir_for_point(p2["x"], p2["y"], cam["R"], cam["T"], fov_deg, image_size)
        n = interpretation_plane_normal(d1, d2)
        if n is None:
            return None
        centers.append(C)
        normals.append(n)
    return triangulate_line(centers, normals)


def estimate_plane_no_mesh(points_by_image: dict, images_sent: list, fov_deg: float, image_size: int,
                           edge_on_thresh: float = EDGE_ON_THRESH_DEFAULT) -> dict:
    """Esquema de 4 pasos: lineas candidatas por par de vistas -> normales
    candidatas por producto cruzado -> puntuar 'de canto' -> refit SVD."""
    view_idxs = sorted(
        int(k) for k, pts in points_by_image.items()
        if get_point_by_obj_id(pts, 1) is not None and get_point_by_obj_id(pts, 2) is not None
    )
    if len(view_idxs) < 4:
        raise ValueError(f"need >=4 valid views (2 independent pairs), got {len(view_idxs)}")

    pair_lines = []
    for i, j in combinations(view_idxs, 2):
        res = _line_from_view_pair(i, j, points_by_image, images_sent, fov_deg, image_size)
        if res is not None:
            point, direction = res
            pair_lines.append(((i, j), point, direction))
    if len(pair_lines) < 2:
        raise ValueError("not enough pair-lines to generate plane candidates")

    view_dirs = {
        idx: view_forward_direction(images_sent[idx]["R"], images_sent[idx]["T"], fov_deg, image_size)
        for idx in view_idxs
    }

    def score_normal(n: np.ndarray) -> float:
        vals = sorted(abs(np.dot(view_dirs[i], n)) for i in view_idxs)
        k = max(2, len(vals) // 2)
        return float(np.mean(vals[:k]))

    candidates = []
    for a in range(len(pair_lines)):
        pa, _, dir_a = pair_lines[a]
        for b in range(a + 1, len(pair_lines)):
            pb, _, dir_b = pair_lines[b]
            if set(pa) & set(pb):
                continue
            n = np.cross(dir_a, dir_b)
            norm = np.linalg.norm(n)
            if norm < 1e-6:
                continue
            candidates.append(n / norm)
    if not candidates:
        raise ValueError("could not generate any candidate normal (pair-lines too parallel)")

    scored = sorted(candidates, key=score_normal)
    best_normal = scored[0]
    good_views = [i for i in view_idxs if abs(np.dot(view_dirs[i], best_normal)) < edge_on_thresh]

    refit_dirs   = [d for (pa, _, d) in pair_lines if set(pa).issubset(good_views)]
    refit_points = [p for (pa, p, _) in pair_lines if set(pa).issubset(good_views)]

    if len(refit_dirs) >= 2:
        D = np.asarray(refit_dirs)
        _, _, Vt = np.linalg.svd(D)
        refined_normal = Vt[-1]
        refined_normal /= np.linalg.norm(refined_normal)
        used_points = refit_points
    else:
        refined_normal = best_normal
        used_points = [p for (_, p, _) in pair_lines]

    origin = np.mean(used_points, axis=0)
    return {
        "normal": refined_normal.tolist(), "origin": origin.tolist(),
        "n_views_used": len(good_views), "n_candidates": len(candidates), "good_views": good_views,
    }


def detect_planes_no_mesh(points_by_image: dict, images_sent: list, fov_deg: float, image_size: int,
                          edge_on_thresh: float = EDGE_ON_THRESH_DEFAULT, max_planes: int = 1,
                          dup_angle_thresh_deg: float = DUP_ANGLE_THRESH_DEFAULT) -> list:
    """Encuentra el plano 1, remueve del pool las vistas 'de canto' que lo
    apoyaron, repite sobre el resto para plano 2, 3, etc."""
    def _ang(v1, v2):
        v1, v2 = v1 / np.linalg.norm(v1), v2 / np.linalg.norm(v2)
        return float(np.degrees(np.arccos(np.clip(np.abs(np.dot(v1, v2)), 0.0, 1.0))))

    all_view_idxs = sorted(
        int(k) for k, pts in points_by_image.items()
        if get_point_by_obj_id(pts, 1) is not None and get_point_by_obj_id(pts, 2) is not None
    )
    pool = set(all_view_idxs)
    planes = []
    while len(planes) < max_planes:
        if len(pool) < 4:
            break
        sub_points = {k: v for k, v in points_by_image.items() if int(k) in pool}
        try:
            pred = estimate_plane_no_mesh(sub_points, images_sent, fov_deg, image_size, edge_on_thresh)
        except ValueError:
            break
        normal = np.array(pred["normal"])
        if any(_ang(normal, np.array(p["normal"])) < dup_angle_thresh_deg for p in planes):
            break
        planes.append(pred)
        pool -= set(pred["good_views"])
    return planes

## 5. Molmo2: carga del modelo + inferencia (inline) -- necesita GPU

Copia funcional de `MolmoPointing/molmo_multiview_runner.py` (Flow A). Sirve
igual para eje y plano -- no depende del symmetry_type, solo de
imagenes+prompt.

In [ ]:
_processor = None
_model     = None


def get_model():
    global _processor, _model
    if _processor is None or _model is None:
        print(f"[model] Loading {MODEL_ID} ...")
        _processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, device_map="auto", use_fast=True)
        _model = AutoModelForImageTextToText.from_pretrained(
            MODEL_ID, trust_remote_code=True, device_map="auto", dtype=torch.bfloat16,
        )
        _model.eval()
        print("[model] Ready.")
    return _processor, _model


def call_model(images: list, prompt: str) -> str:
    processor, model = get_model()
    content = [{"type": "text", "text": prompt}]
    for img in images:
        content.append({"type": "image", "image": img})
    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=2048)
    n_input = inputs["input_ids"].size(1)
    del inputs
    text = processor.tokenizer.decode(output_ids[0, n_input:], skip_special_tokens=True)
    del output_ids
    torch.cuda.empty_cache()
    return text


def parse_single_coords(text: str) -> dict:
    match = re.search(r'coords=["\']([^"\']+)["\']', text)
    if not match:
        return {}
    raw = [float(n) for n in match.group(1).split()]
    if len(raw) < 4:
        return {}
    raw = raw[1:]
    pts = [{"obj_id": int(raw[i]), "x": raw[i + 1], "y": raw[i + 2]} for i in range(0, len(raw) - 2, 3)]
    return {"0": pts} if pts else {}


def parse_multi_coords(text: str, n_images: int) -> dict:
    match = re.search(r'coords=["\']([^"\']+)["\']', text)
    if not match:
        return {}
    result: dict = {}
    for group in match.group(1).split(";"):
        group = group.strip()
        if not group:
            continue
        nums = group.split()
        if len(nums) < 4:
            continue
        try:
            img_idx = int(float(nums[0])) - 1
        except ValueError:
            continue
        if img_idx < 0 or img_idx >= n_images:
            continue
        key, rest, pts = str(img_idx), nums[1:], []
        for i in range(0, len(rest) - 2, 3):
            try:
                pts.append({"obj_id": int(float(rest[i])), "x": float(rest[i + 1]), "y": float(rest[i + 2])})
            except ValueError:
                continue
        if pts:
            result.setdefault(key, []).extend(pts)
    return result


def load_metadata(render_dir: Path) -> list:
    path = render_dir / "metadata_all.json"
    if not path.exists():
        raise FileNotFoundError(f"metadata_all.json not found: {render_dir}")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def get_n_views_entries(metadata: list, n_views: int) -> list:
    total = len(metadata)
    if n_views >= total:
        return sorted(metadata, key=lambda e: e["index"])
    indices = {int(round(i)) for i in np.linspace(0, total - 1, n_views)}
    entries = [m for m in metadata if m["index"] in indices]
    return sorted(entries, key=lambda e: e["index"])


def run_inference(images: list, prompt_single: str, prompt_multi: str):
    """auto mode: 1 imagen -> prompt_single; >1 -> prompt_multi (1 solo llamado)."""
    if len(images) == 1:
        raw = call_model(images, prompt_single)
        pts = parse_single_coords(raw)
        return raw, ({"0": pts["0"]} if "0" in pts else {})
    raw = call_model(images, prompt_multi)
    return raw, parse_multi_coords(raw, n_images=len(images))

## 6. Correr inferencia (eje y plano, por separado)

Guarda `molmo_multiview_<EXPERIMENT_ID>.json` dentro del sandbox. Si el JSON
de una vista ya existe, se saltea -- borralo (o cambia el EXPERIMENT_ID) para
forzar recalculo.

In [ ]:
def run_inference_for_objects(symmetry_type: str, object_list: list, experiment_id: str,
                              prompt_single: str, prompt_multi: str) -> None:
    for cat, oid in object_list:
        render_dir = SANDBOX_RENDERS / symmetry_type / oid / str(SIZE) / LIGHTING
        metadata   = load_metadata(render_dir)
        json_path  = render_dir / f"molmo_multiview_{experiment_id}.json"
        results    = json.load(open(json_path, encoding="utf-8")) if json_path.exists() else {}

        for n_views in VIEW_GROUPS:
            if str(n_views) in results:
                continue
            entries = get_n_views_entries(metadata, n_views)
            images  = [Image.open(render_dir / e["filename"]).convert("RGB") for e in entries]
            raw, points_by_img = run_inference(images, prompt_single, prompt_multi)

            results[str(n_views)] = {
                "experiment_id": experiment_id,
                "prompt_used": prompt_single if n_views == 1 else prompt_multi,
                "raw_output": raw, "points_by_image": points_by_img,
                "images_sent": [
                    {"img_idx": i, "filename": e["filename"], "index": e["index"],
                     "azimuth": e["azimuth"], "elevation": e["elevation"], "eye": e["eye"],
                     "R": e["R"], "T": e["T"]}
                    for i, e in enumerate(entries)
                ],
                "n_points": sum(len(v) for v in points_by_img.values()),
            }
            json_path.parent.mkdir(parents=True, exist_ok=True)
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(results, f, indent=2)
            print(f"  [{symmetry_type:9s}][{cat:6s}] {oid} n_views={n_views}: "
                  f"{results[str(n_views)]['n_points']} puntos devueltos")

In [ ]:
EXPERIMENT_ID_AXIS = "axis_v06_sandbox"

run_inference_for_objects("axis_sym", OBJECT_LIST_AXIS, EXPERIMENT_ID_AXIS,
                          PROMPT_SINGLE_AXIS, PROMPT_MULTI_AXIS)
print("Inferencia de eje completa.")

In [ ]:
EXPERIMENT_ID_PLANE = "plane_v04_1_sandbox"

run_inference_for_objects("plane_sym", OBJECT_LIST_PLANE, EXPERIMENT_ID_PLANE,
                          PROMPT_SINGLE_PLANE, PROMPT_MULTI_PLANE)
print("Inferencia de plano completa.")

## 7. Estimacion sin malla (eje: `triangulation`; plano: `triangulation_multiplane`)

In [ ]:
FILTER_OFF_OBJECT    = False   # True para probar el filtro fondo/objeto (aplica a ambos)
MIN_POINTS_ON_OBJECT = 2

predicted_axes = {}   # (object_id, n_views) -> {"direction", "origin"} | None

for cat, oid in OBJECT_LIST_AXIS:
    render_dir = SANDBOX_RENDERS / "axis_sym" / oid / str(SIZE) / LIGHTING
    manifest_p = render_dir / "manifest.json"
    manifest   = json.load(open(manifest_p, encoding="utf-8")) if manifest_p.exists() else {}
    fov_deg, image_size = manifest.get("fov", DEFAULT_FOV), manifest.get("image_size", SIZE)

    molmo_data  = json.load(open(render_dir / f"molmo_multiview_{EXPERIMENT_ID_AXIS}.json", encoding="utf-8"))
    image_cache = {}
    for n_views_key, group in molmo_data.items():
        points_by_image, images_sent = group["points_by_image"], group["images_sent"]
        if FILTER_OFF_OBJECT:
            points_by_image = filter_points_on_object(
                points_by_image, images_sent, render_dir,
                min_points=MIN_POINTS_ON_OBJECT, image_cache=image_cache,
            )
        try:
            pred = estimate_axis_no_mesh(points_by_image, images_sent, fov_deg, image_size)
        except ValueError as e:
            pred = None
            print(f"  [{cat:6s}] {oid} n_views={n_views_key}: [omitido] {e}")
        predicted_axes[(oid, int(n_views_key))] = pred

print(f"\n[eje] {sum(v is not None for v in predicted_axes.values())}/{len(predicted_axes)} predicciones validas.")

In [ ]:
MAX_PLANES = 3   # igual convencion que --max-planes en estimate_symmetry_no_mesh.py

predicted_planes = {}   # (object_id, n_views) -> lista de planos | []

for cat, oid in OBJECT_LIST_PLANE:
    render_dir = SANDBOX_RENDERS / "plane_sym" / oid / str(SIZE) / LIGHTING
    manifest_p = render_dir / "manifest.json"
    manifest   = json.load(open(manifest_p, encoding="utf-8")) if manifest_p.exists() else {}
    fov_deg, image_size = manifest.get("fov", DEFAULT_FOV), manifest.get("image_size", SIZE)

    molmo_data  = json.load(open(render_dir / f"molmo_multiview_{EXPERIMENT_ID_PLANE}.json", encoding="utf-8"))
    image_cache = {}
    for n_views_key, group in molmo_data.items():
        points_by_image, images_sent = group["points_by_image"], group["images_sent"]
        if FILTER_OFF_OBJECT:
            points_by_image = filter_points_on_object(
                points_by_image, images_sent, render_dir,
                min_points=MIN_POINTS_ON_OBJECT, image_cache=image_cache,
            )
        planes = detect_planes_no_mesh(
            points_by_image, images_sent, fov_deg, image_size,
            edge_on_thresh=EDGE_ON_THRESH_DEFAULT, max_planes=MAX_PLANES,
            dup_angle_thresh_deg=DUP_ANGLE_THRESH_DEFAULT,
        )
        if not planes:
            print(f"  [{cat:6s}] {oid} n_views={n_views_key}: [omitido] no se detecto ningun plano")
        predicted_planes[(oid, int(n_views_key))] = planes

print(f"\n[plano] {sum(len(v) > 0 for v in predicted_planes.values())}/{len(predicted_planes)} objetos con >=1 plano detectado.")

## 8. Evaluacion contra GT (eje y plano)

Copia funcional de `Mapping/evaluate.py::parse_true_label` /
`angular_error_deg` / `point_to_line_distance` / `evaluate_plane_multiset`.

In [ ]:
ANGULAR_THRESHOLDS = [5, 10, 15]


def parse_true_label(txt_path: Path) -> dict:
    """Devuelve {"type": "axis"|"plane", "elements": [...]} -- una malla puede
    tener MAS DE UN plano GT (n_true_planes_mean ~1.15-1.2 en el sweep real)."""
    lines, elements, sym_type = [l.strip() for l in txt_path.read_text().splitlines() if l.strip()], [], None
    for line in lines:
        if line.startswith("axis"):
            sym_type = "axis"
            parts = line.split()
            vec  = np.array([float(x) for x in parts[1:4]]); vec /= np.linalg.norm(vec)
            elements.append({"direction": vec.tolist(), "origin": [float(x) for x in parts[4:7]]})
        elif line.startswith("plane"):
            sym_type = "plane"
            parts = line.split()
            vec  = np.array([float(x) for x in parts[1:4]]); vec /= np.linalg.norm(vec)
            elements.append({"normal": vec.tolist(), "origin": [float(x) for x in parts[4:7]]})
    return {"type": sym_type, "elements": elements}


def angular_error_deg(v1: np.ndarray, v2: np.ndarray) -> float:
    v1, v2 = v1 / np.linalg.norm(v1), v2 / np.linalg.norm(v2)
    return float(np.degrees(np.arccos(np.clip(np.abs(np.dot(v1, v2)), 0.0, 1.0))))


def point_to_line_distance(point: np.ndarray, line_origin: np.ndarray, line_dir: np.ndarray) -> float:
    d = line_dir / np.linalg.norm(line_dir)
    v = point - line_origin
    return float(np.linalg.norm(v - np.dot(v, d) * d))


def evaluate_plane_multiset(pred_planes: list, true_elements: list, angular_threshold_deg: float = 15.0) -> dict:
    """Greedy best-match recall/precision sobre el CONJUNTO COMPLETO de planos GT
    -- Mapping/evaluate.py::evaluate_plane_multiset, copia exacta."""
    n_pred, n_true, matched_gt = len(pred_planes), len(true_elements), set()
    for pred in pred_planes:
        p_normal = np.array(pred["normal"])
        best_idx, best_ang = -1, float("inf")
        for idx, true in enumerate(true_elements):
            if idx in matched_gt:
                continue
            ang = angular_error_deg(p_normal, np.array(true["normal"]))
            if ang < best_ang:
                best_idx, best_ang = idx, ang
        if best_idx >= 0 and best_ang < angular_threshold_deg:
            matched_gt.add(best_idx)
    n_matched = len(matched_gt)
    return {
        "n_planes_predicted": n_pred, "n_true_planes": n_true, "n_planes_matched": n_matched,
        "recall_planes":    round(n_matched / n_true, 4) if n_true else None,
        "precision_planes": round(n_matched / n_pred, 4) if n_pred else None,
    }

In [ ]:
# --- SDE_ref / F1_ref (Mapping/evaluate.py, formulas verbatim) ---
# Requiere gpytoolbox (superficie real via AABB tree) -- disponible en el servidor.
import gpytoolbox as gpy

THRESHOLDS_INLIER    = [0.05, 0.1, 0.15, 0.2]   # metric_F1.py's set_threshold
N_SAMPLES_DEFAULT    = 1000                      # metric_SDE.py's sample count
SDE_REF_SEED_DEFAULT = 0                         # reproducible surface sampling


def sample_surface_points(mesh, n_samples: int = N_SAMPLES_DEFAULT, seed: int | None = SDE_REF_SEED_DEFAULT):
    """Muestra area-weighted de la superficie real -- usada por SDE_ref (eje y plano)."""
    v = np.asarray(mesh.vertices, dtype=np.float64)
    f = np.asarray(mesh.faces, dtype=np.int64)
    if seed is not None:
        np.random.seed(seed)  # el sampler de gpytoolbox usa el RNG global de numpy
    sample = gpy.random_points_on_mesh(v, f, n_samples)
    return v, f, sample


def calaxisloss(axis_dir: np.ndarray, axis_origin: np.ndarray, vertices: np.ndarray,
                faces: np.ndarray, points: np.ndarray) -> float:
    """SDE_ref para eje: refleja puntos de la superficie real 180 grados sobre
    el eje PREDICHO y mide la distancia (al cuadrado) a la superficie real mas
    cercana -- NO usa el GT, es autoconsistencia pura (self-supervised)."""
    t = (points - axis_origin) @ axis_dir
    proj = axis_origin + t[:, None] * axis_dir
    reflected = 2.0 * proj - points
    d, ind, b = gpy.squared_distance(reflected, vertices, faces, use_aabb=True, use_cpp=True)
    return float(np.mean(d))


def calplaneloss(plane: np.ndarray, vertices: np.ndarray, faces: np.ndarray, points: np.ndarray) -> float:
    """SDE_ref para plano: mismo principio, reflejo especular sobre el plano PREDICHO."""
    points = np.hstack((points, np.ones((points.shape[0], 1))))
    lam = points.dot(plane.T)
    planepoints = points - 2 * lam * plane
    d, ind, b = gpy.squared_distance(planepoints[:, 0:3], vertices, faces, use_aabb=True, use_cpp=True)
    return float(np.mean(d))


def normal_origin_to_plane(normal: list, origin: list) -> np.ndarray:
    """Convencion [nx, ny, nz, d] con d = -origin . normal -- la que usa F1_ref."""
    normal = np.asarray(normal, dtype=np.float64)
    origin = np.asarray(origin, dtype=np.float64)
    d = -origin.dot(normal)
    return np.array([normal[0], normal[1], normal[2], d]).reshape(1, 4)


def f1_match_counts(predicted: list, gt_planes: list, threshold_inlier: float) -> tuple[int, int, int]:
    """Copia verbatim de Mapping/evaluate.py::f1_match_counts (greedy, en orden
    de lista -- convencion PRS-Net/E3Sym, NO asignacion optima). El conteo de
    fp por cada gt no-matcheado dentro del loop interno es un detalle real del
    metodo de referencia -- no se "limpia", se replica tal cual."""
    mask = np.zeros(len(gt_planes), dtype=bool)
    tp = fp = 0
    for pred_plane in predicted:
        for idx, gt in enumerate(gt_planes):
            if mask[idx]:
                continue
            val1 = np.linalg.norm(pred_plane - gt)
            val2 = np.linalg.norm(pred_plane + gt)
            val = min(val1, val2)
            if val < threshold_inlier:
                mask[idx] = True
                tp += 1
            else:
                fp += 1
    fn = int(np.sum(~mask))
    return tp, fp, fn


def f1_from_counts(tp: int, fp: int, fn: int) -> float:
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0


print("Nota: F1_ref NO existe para axis_sym (ni el repo de referencia ni la "
      "literatura tienen una convencion de F1 para ejes) -- solo se reporta SDE_ref.")

In [ ]:
mesh_cache_axis = {}   # object_id -> (mesh_v, mesh_f, sample) -- se reusa entre n_views

rows_axis = []
for cat, oid in OBJECT_LIST_AXIS:
    gt = parse_true_label(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["axis_sym"] / f"{oid}.txt")
    t_dir, t_orig = np.array(gt["elements"][0]["direction"]), np.array(gt["elements"][0]["origin"])

    if oid not in mesh_cache_axis:
        mesh = trimesh.load(str(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["axis_sym"] / f"{oid}.obj"), force="mesh", process=False)
        mesh_cache_axis[oid] = sample_surface_points(mesh)
    mesh_v, mesh_f, sample = mesh_cache_axis[oid]

    for n_views in VIEW_GROUPS:
        pred = predicted_axes.get((oid, n_views))
        row = {"categoria": cat, "object_id": oid, "n_views": n_views}
        if pred is None:
            row.update({"status": "no_pred", "angular_error_deg": 90.0, "translation_error": None, "sde_ref": None})
        else:
            p_dir, p_orig = np.array(pred["direction"]), np.array(pred["origin"])
            ang, dist = angular_error_deg(p_dir, t_dir), point_to_line_distance(p_orig, t_orig, t_dir)
            sde = calaxisloss(p_dir, p_orig, mesh_v, mesh_f, sample)
            row.update({"status": "ok", "angular_error_deg": round(ang, 4), "translation_error": round(dist, 6),
                        "sde_ref": round(sde, 8)})
            for t in ANGULAR_THRESHOLDS:
                row[f"precision_{t}deg"] = int(ang < t)
        rows_axis.append(row)

df_eval_axis = pd.DataFrame(rows_axis).sort_values(["categoria", "object_id", "n_views"]).reset_index(drop=True)
df_eval_axis

In [ ]:
mesh_cache_plane = {}   # object_id -> (mesh_v, mesh_f, sample, gt_planes_ref)

rows_plane = []
for cat, oid in OBJECT_LIST_PLANE:
    gt = parse_true_label(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["plane_sym"] / f"{oid}.txt")
    true_elements = gt["elements"]

    if oid not in mesh_cache_plane:
        mesh = trimesh.load(str(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["plane_sym"] / f"{oid}.obj"), force="mesh", process=False)
        mesh_v, mesh_f, sample = sample_surface_points(mesh)
        gt_planes_ref = [normal_origin_to_plane(e["normal"], e["origin"]) for e in true_elements]
        mesh_cache_plane[oid] = (mesh_v, mesh_f, sample, gt_planes_ref)
    mesh_v, mesh_f, sample, gt_planes_ref = mesh_cache_plane[oid]

    for n_views in VIEW_GROUPS:
        planes = predicted_planes.get((oid, n_views), [])
        row = {"categoria": cat, "object_id": oid, "n_views": n_views, "n_true_planes": len(true_elements)}
        if not planes:
            row.update({"status": "no_pred", "recall_planes": 0.0, "precision_planes": None,
                        "n_planes_predicted": 0, "sde_ref": None, "f1_counts": None})
        else:
            m = evaluate_plane_multiset(planes, true_elements, angular_threshold_deg=ANGULAR_THRESHOLDS[-1])
            pred_planes_ref = [normal_origin_to_plane(p["normal"], p["origin"]) for p in planes]
            sde_per_plane = [calplaneloss(pp, mesh_v, mesh_f, sample) for pp in pred_planes_ref]
            f1_counts = {t: f1_match_counts(pred_planes_ref, gt_planes_ref, t) for t in THRESHOLDS_INLIER}
            row.update({"status": "ok", **m,
                        "sde_ref": round(float(np.mean(sde_per_plane)), 8),
                        "f1_counts": f1_counts})
        rows_plane.append(row)

df_eval_plane = pd.DataFrame(rows_plane).sort_values(["categoria", "object_id", "n_views"]).reset_index(drop=True)
# 'f1_counts' es un dict crudo (para agregar F1_ref en la celda siguiente) -- se oculta solo para mostrar la tabla:
df_eval_plane.drop(columns=["f1_counts"])

### 8.1 SDE_ref / F1_ref por conjunto y subconjunto (BUENO/MEDIO/MALO)

`SDE_ref` (eje y plano) se promedia simple dentro de cada grupo. `F1_ref`
(solo plano) acumula TP/FP/FN de TODOS los objetos del grupo por umbral
ANTES de calcular F1 -- no es el promedio de un F1 por objeto (misma
convencion que `Mapping/evaluate.py::_f1_from_counts_by_threshold`; ver
`docs/diagnostico_conditioning_axis.md` para por que esto importa).

In [ ]:
def aggregate_group_metrics(df_axis: pd.DataFrame, df_plane: pd.DataFrame):
    grupos = ["TODOS", "BUENO", "MEDIO", "MALO"]

    axis_rows = []
    for n_views in VIEW_GROUPS:
        for grupo in grupos:
            sub = df_axis[df_axis["n_views"] == n_views]
            if grupo != "TODOS":
                sub = sub[sub["categoria"] == grupo]
            sde_vals = sub["sde_ref"].dropna()
            axis_rows.append({
                "grupo": grupo, "n_views": n_views, "n_objects": len(sub),
                "sde_ref_mean": round(float(sde_vals.mean()), 8) if len(sde_vals) else None,
            })

    plane_rows = []
    for n_views in VIEW_GROUPS:
        for grupo in grupos:
            sub = df_plane[df_plane["n_views"] == n_views]
            if grupo != "TODOS":
                sub = sub[sub["categoria"] == grupo]
            sde_vals = sub["sde_ref"].dropna()

            totals = {t: [0, 0, 0] for t in THRESHOLDS_INLIER}
            for counts in sub["f1_counts"].dropna():
                for t in THRESHOLDS_INLIER:
                    tp, fp, fn = counts[t]
                    totals[t][0] += tp; totals[t][1] += fp; totals[t][2] += fn
            has_any = any(sum(totals[t]) > 0 for t in THRESHOLDS_INLIER)
            f1_per_t = [f1_from_counts(*totals[t]) for t in THRESHOLDS_INLIER]

            plane_rows.append({
                "grupo": grupo, "n_views": n_views, "n_objects": len(sub),
                "sde_ref_mean": round(float(sde_vals.mean()), 8) if len(sde_vals) else None,
                "f1_ref": round(float(np.mean(f1_per_t)), 4) if has_any else None,
            })

    return pd.DataFrame(axis_rows), pd.DataFrame(plane_rows)


axis_group_metrics, plane_group_metrics = aggregate_group_metrics(df_eval_axis, df_eval_plane)

print("=== SDE_ref por grupo x n_views -- EJE ===")
display(axis_group_metrics)
print("\n=== SDE_ref / F1_ref por grupo x n_views -- PLANO ===")
display(plane_group_metrics)

## 9. Resultados: resumen por categoria y por n_views

In [ ]:
print(f"=== EJE -- {EXPERIMENT_ID_AXIS}  |  filter_off_object={FILTER_OFF_OBJECT} ===\n")
print("--- Promedio por categoria x n_views ---")
display(df_eval_axis.groupby(["categoria", "n_views"])[["angular_error_deg"]].mean().round(2))
print("\n--- Promedio global por n_views ---")
display(df_eval_axis.groupby("n_views")[["angular_error_deg"]].agg(["mean", "median", "std", "min", "max"]).round(2))

In [ ]:
print(f"=== PLANO -- {EXPERIMENT_ID_PLANE}  |  max_planes={MAX_PLANES}  |  filter_off_object={FILTER_OFF_OBJECT} ===\n")
print("--- Promedio por categoria x n_views ---")
display(df_eval_plane.groupby(["categoria", "n_views"])[["recall_planes", "precision_planes"]].mean().round(3))
print("\n--- Promedio global por n_views ---")
display(df_eval_plane.groupby("n_views")[["recall_planes", "precision_planes"]].agg(["mean", "median", "std"]).round(3))

## 10. Visualizacion 2D/3D (opcional)

Reescrita inline (sin importar nada del repo salvo `trimesh`/`PIL`/`matplotlib`,
ya cargados en el Setup). Un caso de eje y un caso de plano, a modo de ejemplo.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


def _plot_2d(symmetry_type: str, object_id: str, categoria: str, experiment_id: str,
            n_views: int, subtitle: str, max_views_2d: int = 6) -> None:
    render_dir = SANDBOX_RENDERS / symmetry_type / object_id / str(SIZE) / LIGHTING
    molmo_data = json.load(open(render_dir / f"molmo_multiview_{experiment_id}.json", encoding="utf-8"))
    group = molmo_data[str(n_views)]
    images_sent, points_by_image = group["images_sent"], group["points_by_image"]

    idxs = sorted(points_by_image.keys(), key=int)[:max_views_2d]
    fig, axes = plt.subplots(1, len(idxs), figsize=(3.2 * len(idxs), 3.4))
    if len(idxs) == 1:
        axes = [axes]
    for ax, idx_str in zip(axes, idxs):
        cam = images_sent[int(idx_str)]
        img_path = render_dir / cam["filename"]
        img = np.array(Image.open(img_path)) if img_path.exists() else None
        if img is not None:
            ax.imshow(img)
            img_h, img_w = img.shape[0], img.shape[1]
        else:
            img_w = img_h = SIZE
            ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
        for p in points_by_image[idx_str]:
            px, py = molmo_xy_to_pixel(p["x"], p["y"], img_w, img_h)
            ax.scatter([px], [py], c="red" if p["obj_id"] == 1 else "blue",
                       s=60, edgecolors="white", linewidths=1.2, zorder=5)
        ax.set_title(f"img {idx_str}", fontsize=8)
        ax.axis("off")
    fig.suptitle(f"[{categoria}] {object_id} -- {subtitle}", fontsize=10)
    fig.tight_layout()
    plt.show()


def plot_axis_case(object_id: str, categoria: str, n_views: int) -> None:
    pred = predicted_axes.get((object_id, n_views))
    gt   = parse_true_label(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["axis_sym"] / f"{object_id}.txt")
    err  = df_eval_axis[(df_eval_axis.object_id == object_id) & (df_eval_axis.n_views == n_views)]["angular_error_deg"].iloc[0]
    _plot_2d("axis_sym", object_id, categoria, EXPERIMENT_ID_AXIS, n_views, f"angular_error={err:.2f} grados")

    mesh  = trimesh.load(str(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["axis_sym"] / f"{object_id}.obj"), force="mesh", process=False)
    verts = np.asarray(mesh.vertices)
    verts_plot = verts if len(verts) <= 4000 else verts[np.random.default_rng(0).choice(len(verts), 4000, replace=False)]
    bbox_diag = float(np.linalg.norm(verts.max(axis=0) - verts.min(axis=0)))

    fig = plt.figure(figsize=(6.5, 6.5))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(verts_plot[:, 0], verts_plot[:, 1], verts_plot[:, 2], s=1, c="lightgray", alpha=0.4, label="malla")

    def plot_line(origin, direction, color, label):
        d = np.array(direction) / np.linalg.norm(direction)
        o = np.array(origin)
        p0, p1 = o - d * bbox_diag * 0.75, o + d * bbox_diag * 0.75
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], [p0[2], p1[2]], color=color, linewidth=2.5, label=label)

    plot_line(gt["elements"][0]["origin"], gt["elements"][0]["direction"], "green", "eje GT")
    if pred is not None:
        plot_line(pred["origin"], pred["direction"], "red", "eje predicho")
    ax.set_title(f"[{categoria}] {object_id} -- angular_error={err:.2f} grados", fontsize=10)
    ax.legend(loc="upper left", fontsize=8)
    ax.set_box_aspect([1, 1, 1])
    fig.tight_layout()
    plt.show()


def _plane_patch(origin, normal, size, color, label, ax):
    """Dibuja el plano como un cuadrado finito centrado en origin, para poder verlo en 3D."""
    normal = np.array(normal) / np.linalg.norm(normal)
    origin = np.array(origin)
    tmp = np.array([1.0, 0.0, 0.0]) if abs(normal[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    u = np.cross(normal, tmp); u /= np.linalg.norm(u)
    v = np.cross(normal, u)
    corners = [origin + s1 * size * u + s2 * size * v for s1 in (-1, 1) for s2 in (-1, 1)]
    corners = [corners[0], corners[1], corners[3], corners[2]]  # orden para que el poligono no se cruce
    xs, ys, zs = zip(*corners)
    ax.plot(list(xs) + [xs[0]], list(ys) + [ys[0]], list(zs) + [zs[0]], color=color, linewidth=2, label=label)


def plot_plane_case(object_id: str, categoria: str, n_views: int) -> None:
    planes = predicted_planes.get((object_id, n_views), [])
    gt     = parse_true_label(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["plane_sym"] / f"{object_id}.txt")
    row    = df_eval_plane[(df_eval_plane.object_id == object_id) & (df_eval_plane.n_views == n_views)].iloc[0]
    _plot_2d("plane_sym", object_id, categoria, EXPERIMENT_ID_PLANE, n_views,
             f"recall={row['recall_planes']}, precision={row['precision_planes']}")

    mesh  = trimesh.load(str(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["plane_sym"] / f"{object_id}.obj"), force="mesh", process=False)
    verts = np.asarray(mesh.vertices)
    verts_plot = verts if len(verts) <= 4000 else verts[np.random.default_rng(0).choice(len(verts), 4000, replace=False)]
    bbox_diag = float(np.linalg.norm(verts.max(axis=0) - verts.min(axis=0)))

    fig = plt.figure(figsize=(6.5, 6.5))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(verts_plot[:, 0], verts_plot[:, 1], verts_plot[:, 2], s=1, c="lightgray", alpha=0.4, label="malla")

    for i, e in enumerate(gt["elements"]):
        _plane_patch(e["origin"], e["normal"], bbox_diag * 0.4, "green", f"plano GT {i+1}" if i == 0 else None, ax)
    for i, p in enumerate(planes):
        _plane_patch(p["origin"], p["normal"], bbox_diag * 0.4, "red", f"plano predicho {i+1}" if i == 0 else None, ax)

    ax.set_title(f"[{categoria}] {object_id} -- recall={row['recall_planes']}, precision={row['precision_planes']}", fontsize=10)
    ax.legend(loc="upper left", fontsize=8)
    ax.set_box_aspect([1, 1, 1])
    fig.tight_layout()
    plt.show()


# Ejemplo: un BUENO y un MALO de cada tipo, al mayor n_views
plot_axis_case(OBJECT_LIST_AXIS[0][1], "BUENO", n_views=max(VIEW_GROUPS))
plot_axis_case(OBJECT_LIST_AXIS[-1][1], "MALO", n_views=max(VIEW_GROUPS))
plot_plane_case(OBJECT_LIST_PLANE[0][1], "BUENO", n_views=max(VIEW_GROUPS))
plot_plane_case(OBJECT_LIST_PLANE[-1][1], "MALO", n_views=max(VIEW_GROUPS))